# Free Colab: select and fine-tune a native essay scorer
Choose **Runtime → Change runtime type → T4 GPU**, then Run all. Upload **recovered-model-comparison.zip** when asked. No paid GPU or Claude key is required.

This is a focused scoring experiment, not a finished teacher-calibrated grader. Compare Qwen3-8B and Qwen2.5-7B-Instruct on validation scores; train the measured winner using human ASAP 2.0 holistic scores. Best means best among these candidates under this input contract. Free GPU availability and session duration are controlled by Colab; completion in a single session is not guaranteed.

All 186 supplied/prior cases are imported unchanged, including Amplify, the custom rubric, narrative examples, style preferences and human commentary. They are excluded from new training and retained for review; Amplify's 1–4 scores receive a separate development check. This run deliberately does **not** turn incompatible rubric labels or qualitative commentary into invented 1–6 training labels. It trains native scoring, not narrative feedback or six-category rubric ratings.

Five ASAP prompts train the adapter; an electoral-college prompt selects candidates/checkpoints; driverless-cars essays remain reserved until training ends. Whole essays and the native rubric are retained. Source context uses at most 300 retrieved words, so comprehensive source verification is not assessed. OCR source passages have not been human proofread. This balanced small test is exploratory and does not establish classroom accuracy.


In [ ]:
import base64,hashlib,io,json,pathlib,sys,subprocess,zipfile
from datetime import datetime,timezone
root=pathlib.Path('/content')/('focused-essay-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
root.mkdir(parents=True,exist_ok=False)
snapshot=base64.b64decode('...')
assert hashlib.sha256(snapshot).hexdigest()=='5c6036cf155c3ba4d11c015c89199f0f470b4f2724235d936181e5e2bd59faca'
with zipfile.ZipFile(io.BytesIO(snapshot)) as z:
    for name in z.namelist():
        target=(root/name).resolve()
        if not target.is_relative_to(root.resolve()):raise RuntimeError('Invalid code path')
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))
sys.path.insert(0,str(root))
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(root/'requirements-focused-training.txt')],check=True)
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','poppler-utils','tesseract-ocr'],check=True)
import torch
if not torch.cuda.is_available():raise RuntimeError('Select a T4 GPU runtime, then rerun.')
print('GPU:',torch.cuda.get_device_name(0));print('Run folder:',root)
(root/'code-snapshot.json').write_text(json.dumps({'sha256':hashlib.sha256(snapshot).hexdigest(),'files':['model_work/__init__.py', 'model_work/baseline_data.py', 'model_work/baseline_runner.py', 'model_work/focused_scoring.py', 'model_work/focused_engine.py', 'requirements-focused-training.txt']},indent=2))


In [ ]:
from google.colab import files
from model_work.focused_scoring import import_previous,prepare,preflight,CANDIDATES,score_metrics,messages
from model_work.baseline_data import collect_asap,collect_foxglove
uploaded=files.upload()
archives=[pathlib.Path(n) for n in uploaded if n.endswith('.zip')]
if len(archives)!=1:raise RuntimeError('Upload only recovered-model-comparison.zip.')
previous=import_previous(archives[0],root/'previous')
del uploaded
print('Preserved',len(previous),'prior cases, including Amplify and the supplied rubric.')
corpus,sources,rubric=collect_asap(root/'asap2')
foxglove=collect_foxglove(root/'foxglove')


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer
api=HfApi();specs=[];tokenizers={}
for model_id in CANDIDATES:
    revision=api.model_info(model_id).sha
    spec={'model':model_id,'revision':revision,'name':model_id.split('/')[-1]}
    specs.append(spec)
    tokenizers[model_id]=AutoTokenizer.from_pretrained(model_id,revision=revision,trust_remote_code=False)
(root/'model-pins.json').write_text(json.dumps(specs,indent=2))
manifest=prepare(corpus,sources,rubric,previous,foxglove,list(tokenizers.values()),root/'dataset',
                 max_length=3072,train_per_cell=12,eval_per_score=4)
print(json.dumps(manifest['counts'],indent=2))
print('No essays are silently truncated. Actual eligible counts can be below the caps.')


In [ ]:
from model_work.focused_engine import select,train,evaluate
selection=select(root/'dataset',specs,tokenizers,root/'selection')
selected=selection['selected'];tokenizer=tokenizers[selected['model']]
print('Selected:',selected['model']);print(json.dumps(selection,indent=2))


The next cell trains only native score + end-of-answer tokens. Rank-8 QLoRA, batch size one, gradient checkpointing and tail-only logits reduce GPU memory use. The best epoch is selected by validation MAE. Reserved test performance never changes training settings in this run. If both candidates fail validation, the notebook stops instead of calling either one the winner.

In [ ]:
model,training_summary=train(root/'dataset',selected,tokenizer,root/'training',epochs=2)
# Save a compact, usable adapter before running any test or optional API call.
import shutil
checkpoint=root/'checkpoint-export';checkpoint.mkdir()
shutil.copytree(root/'training'/'adapter',checkpoint/'adapter')
for src in (root/'training'/'training-summary.json',root/'model-pins.json',root/'dataset'/'manifest.json',root/'code-snapshot.json',root/'selection'/'selection.json'):
    shutil.copy2(src,checkpoint/src.name)
checkpoint_zip=shutil.make_archive(str(root/'focused-score-checkpoint'),'zip',checkpoint)
print('Adapter ZIP size MiB:',round(pathlib.Path(checkpoint_zip).stat().st_size/2**20,2))
files.download(checkpoint_zip)


In [ ]:
manifest,splits=preflight(root/'dataset')
results=root/'results';results.mkdir()
# This is the first model evaluation using reserved test labels.
with model.disable_adapter():
    base_metrics=evaluate(model,tokenizer,splits['test'],manifest['max_length'],results/'base.test.jsonl')
tuned_metrics=evaluate(model,tokenizer,splits['test'],manifest['max_length'],results/'tuned.test.jsonl')
comparison={'base':base_metrics,'tuned':tuned_metrics,'contract':'same retrieved-context, native score-only task',
            'deployment_approved':False,'small_balanced_prompt_holdout':True}
print(json.dumps(comparison,indent=2))


In [ ]:
# Amplify is a separate, previously seen development check on its ORIGINAL 1–4 scale.
amplify=[]
for case in previous:
    if case['track']!='amplify_native':continue
    content=json.loads(case['input_messages'][1]['content'])
    amplify.append({'id':case['id'],'writing':case['writing'],'score':case['reference']['score'],
        'grade':content.get('grade'),'assignment':content.get('assignment'),'rubric':content.get('rubric'),
        'native_criterion':content.get('criterion'),'source_excerpts':None,'score_min':1,'score_max':4,
        'genre':case.get('genre')})
# No clipping for longer auxiliary inputs; failures remain visible in coverage.
with model.disable_adapter():
    comparison['amplify_development_base']=evaluate(model,tokenizer,amplify,manifest['max_length'],results/'amplify.base.jsonl')
comparison['amplify_development_tuned']=evaluate(model,tokenizer,amplify,manifest['max_length'],results/'amplify.tuned.jsonl')
for genre in sorted({r['genre'] for r in amplify}):
    rows=[r for r in amplify if r['genre']==genre]
    pred=[json.loads(s) for s in (results/'amplify.tuned.jsonl').read_text().splitlines()]
    comparison['amplify_tuned_'+genre]=score_metrics(rows,[p for p in pred if p['id'] in {r['id'] for r in rows}])
print(json.dumps(comparison,indent=2))


Optional Claude comparison below: leave the key blank to skip. Supplying a key makes about one call per reserved test essay and uses your Anthropic billing. This uses the same new score-only inputs as this run; the earlier 186-case Claude results remain a different task and must not be treated as a direct comparison.

In [ ]:
from getpass import getpass
from model_work.baseline_runner import claude_text
key=getpass('Optional Anthropic key (blank skips): ').strip()
if key:
    import anthropic,re
    attempts=[]
    with anthropic.Anthropic(api_key=key,timeout=120,max_retries=2) as client:
        for i,row in enumerate(splits['test']):
            try:
                reply=claude_text(client,'claude-haiku-4-5',messages(row),max_tokens=16).strip()
                if not re.fullmatch('[1-6]',reply):raise ValueError('Noninteger response')
                attempts.append({'id':row['id'],'score':int(reply)})
            except Exception as exc:
                if i==0:raise RuntimeError('Claude setup failed: '+type(exc).__name__+'. No remaining calls made.') from None
                attempts.append({'id':row['id'],'score':None,'error_type':type(exc).__name__})
    (results/'claude.test.jsonl').write_text(''.join(json.dumps(p)+'\n' for p in attempts))
    comparison['claude']=score_metrics(splits['test'],attempts)
    comparison['claude_model']='claude-haiku-4-5'
else:comparison['claude']='skipped; no key supplied'
key=''


In [ ]:
# Export review evidence even when optional Claude is skipped.
(results/'comparison.json').write_text(json.dumps(comparison,indent=2))
for src in (root/'training'/'training-summary.json',root/'model-pins.json',root/'dataset'/'manifest.json',root/'code-snapshot.json'):
    shutil.copy2(src,results/src.name)
shutil.copytree(root/'selection',results/'selection')
shutil.copytree(root/'previous',results/'previous-material')
# Include held-out inputs/labels for independent review; no raw full training corpus in export.
shutil.copy2(root/'dataset'/'test.jsonl',results/'reserved-test-cases.jsonl')
for folder in ('asap2','foxglove'):
    shutil.copy2(root/folder/'provenance.json',results/(folder+'-provenance.json'))
shutil.copy2(root/'asap2'/'source-extraction-methods.json',results/'source-extraction-methods.json')
(results/'README.txt').write_text('Experimental native score adapter. Not approved for deployment.\nASAP 2.0: Crossley et al. (2025), CC-BY-4.0; see provenance.\nFOXGLOVE reference corpus: see provenance and original license. User-supplied material stays in private results.\nAll prior cases are retained separately; only human ASAP holistic scores train this run.\nScore probabilities are relative choice probabilities, not calibrated confidence.\n')
result_zip=shutil.make_archive(str(root/'focused-score-results'),'zip',results)
files.download(result_zip)
print('Finished. Return focused-score-results.zip for review; keep focused-score-checkpoint.zip safe.')


Keep both downloads. Upload **focused-score-results.zip** back into chat so we can assess improvement, score collapse and Amplify/narrative regressions. Keep the adapter ZIP for implementation after review. This notebook has no site deployment step. If interrupted after training, the adapter export cell can be run before optional comparisons; preserve the runtime folder until downloads finish.